## 00 · Data Audit — COSMO belief–affect–behavior network pipeline

### Codebook-based measurement notes (variables used in this thesis)

This audit uses the following COSMO variables (see codebook) and applies
measurement-aware handling for availability checks and later preprocessing:

- **Wave identifier**
  - `TIME`: wave number (used for grouping).

- **Focal belief**
  - `SEVERITY` (1–7): perceived severity of a COVID-19 infection  
    (`1 = völlig harmlos` … `7 = extrem gefährlich`).

- **Affect**
  - `AFF_FEAR` (1–7): *reverse-keyed* (`1 = angsteinflößend` … `7 = nicht angsteinflößend`).  
  - `AFF_WORRY` (1–7): *reverse-keyed* (`1 = besorgnis erregend` … `7 = nicht besorgnis erregend`).  
  - `WORRY_HEALTH_SYSTEM` (1–7): worry about health-system overload  
    (`1 = sehr wenig Sorgen` … `7 = sehr viele Sorgen`).  
  - `AFF_FEAR` and `AFF_WORRY` are reverse-coded in **01_preprocessing** so that  
    higher values consistently reflect more negative affect / greater perceived threat.

- **Behavior** (individual protective measures; ordinal frequency)
  - `USE2_MASK`, `USE2_SPACE150`, `USE2_HANDWASH20`, `USE2_AVOID` (1–6):  
    `1 = Trifft nicht zu`, `2 = Nie`, `3 = Selten`,  
    `4 = Manchmal`, `5 = Häufig`, `6 = Immer`.  
  - Category `1 = Trifft nicht zu` is treated as **non-applicable** and recoded to  
    missing for this audit (and all subsequent analyses); substantive range is **2–6**.  
  - ⚠ `USE2_MASK` wording changes at **wave 22** ("Atemschutzmaske tragen" →  
    "Maske getragen"). This is flagged below; the selected waves must respect this  
    boundary or the boundary must be explicitly acknowledged.

- **Trust descriptor** (context variable; *not* a network node)
  - `TRUST_RKI` (3–9 substantive; see notes):  
    `1 = keine Angabe möglich` → recoded to missing.  
    **Note:** The codebook indicates substantive responses begin at **3**.  
    Value `2`, if present, should also be treated as non-substantive and recoded  
    to missing. This is handled explicitly below (see Cell 2).

- **Wave inclusion criterion**
  - A wave is retained if every audited variable has **≥ 90 % non-missing**  
    responses *after* codebook-based recoding.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# IPython display — falls back gracefully when run outside Jupyter
try:
    from IPython.display import display
except ImportError:
    display = print

# ── Load data ─────────────────────────────────────────────────────────────────
# Use pathlib for cross-platform path handling (forward slashes work on all OS)
DATA_PATH = Path("..") / "data" / "raw_data_COSMO.csv"

df = pd.read_csv(DATA_PATH)

print("Loaded df shape:", df.shape)
print("Number of columns:", len(df.columns))

Loaded df shape: (70010, 334)
Number of columns: 334


In [2]:
# ── Wave column ───────────────────────────────────────────────────────────────
WAVE = "TIME"
if WAVE not in df.columns:
    raise ValueError(f"'{WAVE}' not found in df.columns")

# Ensure wave codes are numeric; non-parseable entries become NaN
df[WAVE] = pd.to_numeric(df[WAVE], errors="coerce")

# ── Variable lists ────────────────────────────────────────────────────────────
FOCAL_BELIEF   = "SEVERITY"
AFFECT_VARS    = ["AFF_FEAR", "AFF_WORRY", "WORRY_HEALTH_SYSTEM"]
BEHAVIOR_VARS  = ["USE2_MASK", "USE2_SPACE150", "USE2_HANDWASH20", "USE2_AVOID"]
TRUST_VARS     = ["TRUST_RKI"]   # context descriptor — not a network node

VARIABLES      = [FOCAL_BELIEF] + AFFECT_VARS + BEHAVIOR_VARS + TRUST_VARS

# ── Verify all required columns are present ───────────────────────────────────
REQ_COLS    = [WAVE] + VARIABLES
missing_cols = [c for c in REQ_COLS if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# ── 1. Replace common numeric sentinel codes with NaN ────────────────────────
# These cover typical SPSS / survey export conventions.
MISSING_CODES = [
    -99, -98, -97, -96, -95, -94, -93, -92, -91, -90,
    -9,  -8,  -7,  -6,  -5,  -4,  -3,  -2,  -1,
    95, 96, 97, 98, 99,
    999, 9999,
]
df[VARIABLES] = df[VARIABLES].replace(MISSING_CODES, np.nan)

# ── 2. Replace common string-based missing tokens with NaN ───────────────────
STRING_MISSING = ["", " ", "NA", "N/A", "nan", "NaN", "NULL", "null",
                  "keine Angabe", "weiß nicht", "weiss nicht"]
for v in VARIABLES:
    if df[v].dtype == object:
        df[v] = df[v].replace(STRING_MISSING, np.nan)

# ── 3. Codebook-specific non-substantive recodes ─────────────────────────────
# TRUST_RKI:
#   1 = "keine Angabe möglich" → missing (codebook-explicit)
#   2 → missing if present (codebook implies substantive range starts at 3;
#       inspect the scale_report output in Cell 5 to verify whether 2 appears)
_trust_nonsub = [1, 2]
df.loc[df["TRUST_RKI"].isin(_trust_nonsub), "TRUST_RKI"] = np.nan

# USE2_* behavior items:
#   1 = "Trifft nicht zu" (not applicable) → missing
for v in BEHAVIOR_VARS:
    df.loc[df[v].eq(1), v] = np.nan

# ── Confirmation ──────────────────────────────────────────────────────────────
print("WAVE column :", WAVE)
print("Node variables :", [FOCAL_BELIEF] + AFFECT_VARS + BEHAVIOR_VARS)
print("Context variable :", TRUST_VARS)
print("TRUST_RKI non-substantive codes recoded to NaN:", _trust_nonsub)
print("USE2_* category 1 ('Trifft nicht zu') recoded to NaN for all behavior variables.")

WAVE column : TIME
Node variables : ['SEVERITY', 'AFF_FEAR', 'AFF_WORRY', 'WORRY_HEALTH_SYSTEM', 'USE2_MASK', 'USE2_SPACE150', 'USE2_HANDWASH20', 'USE2_AVOID']
Context variable : ['TRUST_RKI']
TRUST_RKI non-substantive codes recoded to NaN: [1, 2]
USE2_* category 1 ('Trifft nicht zu') recoded to NaN for all behavior variables.


In [3]:
# ── Wave-level availability audit ────────────────────────────────────────────
# Retain waves where EVERY audited variable has >= 90 % non-missing responses
# after the codebook-based recodes applied in Cell 2.

THRESH = 0.90

rows = []
for w, d in df.groupby(WAVE):
    rates  = {v: float(d[v].notna().mean()) for v in VARIABLES}
    passed = all(r >= THRESH for r in rates.values())
    rows.append({
        "wave"   : int(w) if pd.notna(w) else w,
        "n_wave" : int(len(d)),
        "pass_90": passed,
        **{f"avail_{v}": r for v, r in rates.items()},
    })

wave_avail  = pd.DataFrame(rows).sort_values("wave").reset_index(drop=True)
waves_keep  = wave_avail.loc[wave_avail["pass_90"], "wave"].astype(int).tolist()

print(f"Availability threshold : {THRESH:.0%}")
print(f"Waves passing threshold: {waves_keep}")
print()

display(wave_avail)

Availability threshold : 90%
Waves passing threshold: [9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 30, 31, 32, 33, 34, 35, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 54, 55, 56, 57, 58]



,wave,n_wave,pass_90,avail_SEVERITY,avail_AFF_FEAR,avail_AFF_WORRY,avail_WORRY_HEALTH_SYSTEM,avail_USE2_MASK,avail_USE2_SPACE150,avail_USE2_HANDWASH20,avail_USE2_AVOID,avail_TRUST_RKI
0,1,973,False,1.0,1.0,1.0,0.0,0.000000,0.000000,0.0,0.0,0.915725
1,2,966,False,1.0,1.0,1.0,0.0,0.000000,0.000000,0.0,0.0,0.939959
2,3,1015,False,1.0,1.0,1.0,1.0,0.000000,0.000000,0.0,0.0,0.964532
3,4,956,False,1.0,1.0,1.0,1.0,0.000000,0.000000,0.0,0.0,0.967573
4,5,1028,False,1.0,1.0,1.0,1.0,0.000000,0.000000,0.0,0.0,0.974708
...,...,...,...,...,...,...,...,...,...,...,...,...
65,67,998,False,1.0,1.0,1.0,1.0,0.949900,0.956914,0.0,0.0,0.974950
66,68,991,False,1.0,1.0,1.0,1.0,0.938446,0.940464,0.0,0.0,0.965691
67,69,953,False,1.0,1.0,1.0,1.0,0.929696,0.924449,0.0,0.0,0.967471
68,70,1003,False,1.0,1.0,1.0,1.0,0.932203,0.922233,0.0,0.0,0.972084


In [4]:
# ── Restrict data to candidate waves ─────────────────────────────────────────
waves_candidate = waves_keep

waves_df = df.loc[
    df[WAVE].isin(waves_candidate),
    [WAVE] + VARIABLES
].copy()

print("waves_df shape     :", waves_df.shape)
print("Candidate waves    :", waves_candidate)
print("Unique waves found :", sorted(waves_df[WAVE].dropna().unique()))
print()

display(waves_df.head())

waves_df shape     : (46774, 10)
Candidate waves    : [9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 30, 31, 32, 33, 34, 35, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 54, 55, 56, 57, 58]
Unique waves found : [np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(37), np.int64(38), np.int64(39), np.int64(40), np.int64(41), np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(49), np.int64(50), np.int64(51), np.int64(52), np.int64(54), np.int64(55), np.int64(56), np.int64(57), np.int64(58)]



,TIME,SEVERITY,AFF_FEAR,AFF_WORRY,WORRY_HEALTH_SYSTEM,USE2_MASK,USE2_SPACE150,USE2_HANDWASH20,USE2_AVOID,TRUST_RKI
7998,9,3.0,3.0,2.0,1.0,4.0,6.0,6.0,6.0,8.0
7999,9,4.0,4.0,5.0,1.0,6.0,6.0,6.0,5.0,9.0
8000,9,3.0,3.0,3.0,6.0,6.0,6.0,6.0,NaN,7.0
8001,9,7.0,1.0,1.0,7.0,5.0,6.0,6.0,5.0,9.0
8002,9,6.0,7.0,7.0,4.0,6.0,6.0,6.0,6.0,6.0


In [5]:
# ── Scale diagnostics on candidate-wave data ─────────────────────────────────
# Reports the empirical response range after codebook recoding.
# 'any_outside_expected' is variable-aware:
#   - most variables: valid range 1–7
#   - TRUST_RKI: valid substantive range 3–9 (after recoding 1,2 → NaN)
#   - USE2_*: valid substantive range 2–6 (after recoding 1 → NaN)

EXPECTED_RANGES = {
    "SEVERITY"           : (1, 7),
    "AFF_FEAR"           : (1, 7),
    "AFF_WORRY"          : (1, 7),
    "WORRY_HEALTH_SYSTEM": (1, 7),
    "USE2_MASK"          : (2, 6),
    "USE2_SPACE150"      : (2, 6),
    "USE2_HANDWASH20"    : (2, 6),
    "USE2_AVOID"         : (2, 6),
    "TRUST_RKI"          : (3, 9),
}

audit_df = df.loc[df[WAVE].isin(waves_candidate), [WAVE] + VARIABLES].copy()

def scale_report(d, var):
    x     = pd.to_numeric(d[var], errors="coerce").dropna()
    lo, hi = EXPECTED_RANGES.get(var, (1, 7))

    return {
        "var"                  : var,
        "expected_range"       : f"{lo}–{hi}",
        "non_missing_n"        : int(x.shape[0]),
        "observed_min"         : float(x.min())  if len(x) else np.nan,
        "observed_max"         : float(x.max())  if len(x) else np.nan,
        "n_unique"             : int(x.nunique()) if len(x) else 0,
        "unique_values"        : sorted(x.unique().tolist())[:20],
        "any_outside_expected" : bool(((x < lo) | (x > hi)).any()) if len(x) else False,
        "any_zero"             : bool((x == 0).any())               if len(x) else False,
    }

reports = pd.DataFrame([scale_report(audit_df, v) for v in VARIABLES])
display(reports)

# Flag any variable with unexpected values
flagged = reports[reports["any_outside_expected"] | reports["any_zero"]]
if flagged.empty:
    print("✓ No out-of-range values detected.")
else:
    print("⚠ Variables with out-of-range or zero values:")
    display(flagged[["var", "expected_range", "observed_min", "observed_max", "unique_values"]])

,var,expected_range,non_missing_n,observed_min,observed_max,n_unique,unique_values,any_outside_expected,any_zero
0,SEVERITY,1–7,46774,1.0,7.0,7,"[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]",False,False
1,AFF_FEAR,1–7,46774,1.0,7.0,7,"[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]",False,False
2,AFF_WORRY,1–7,46774,1.0,7.0,7,"[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]",False,False
3,WORRY_HEALTH_SYSTEM,1–7,46774,1.0,7.0,7,"[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]",False,False
4,USE2_MASK,2–6,46264,2.0,6.0,5,"[2.0, 3.0, 4.0, 5.0, 6.0]",False,False
5,USE2_SPACE150,2–6,46089,2.0,6.0,5,"[2.0, 3.0, 4.0, 5.0, 6.0]",False,False
6,USE2_HANDWASH20,2–6,45993,2.0,6.0,5,"[2.0, 3.0, 4.0, 5.0, 6.0]",False,False
7,USE2_AVOID,2–6,44658,2.0,6.0,5,"[2.0, 3.0, 4.0, 5.0, 6.0]",False,False
8,TRUST_RKI,3–9,45574,0.0,9.0,8,"[0.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0]",True,True


⚠ Variables with out-of-range or zero values:


,var,expected_range,observed_min,observed_max,unique_values
8,TRUST_RKI,3–9,0.0,9.0,"[0.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0]"


## Wave quality ranking — selecting the best 3-wave block

Passing the 90 % availability threshold is a necessary but not sufficient  
condition for informative network estimation.  Behavioral nodes can become  
isolated when (i) responses are near-constant (low variance) and/or  
(ii) behavior is weakly associated with the belief–affect set, causing  
EBICglasso regularization to shrink all edges to zero.

We therefore compute a composite wave-quality score for each candidate wave  
from three **standardized** components:

| Component | Rationale |
|-----------|-----------|
| Complete-case *N* (`n_cc`) | Statistical power for GGM estimation |
| Mean Shannon entropy across behavior items (nats, categories 2–6) | Information content in behavioral responses |
| Mean of the maximum absolute Spearman ρ between each behavior item and any belief/affect variable | Behavior–psychology coupling (estimation-relevant signal) |

**Standardization:** Each component is z-scored across candidate waves before  
summation so that sample size does not mechanically dominate the score.

To ensure temporal comparability and minimize confounds from pandemic-context  
shifts, waves are selected as the highest-scoring **consecutive 3-wave block**  
(i.e., waves *w*, *w+1*, *w+2*).

⚠ The selected block must **not straddle wave 22**, the boundary at which the  
`USE2_MASK` item wording changes.  This constraint is enforced below.

In [6]:
# ── Wave quality ranking ──────────────────────────────────────────────────────
from scipy.stats import zscore   # requires scipy; part of standard scientific stack

NODE_VARS  = [FOCAL_BELIEF] + AFFECT_VARS + BEHAVIOR_VARS
BEH_LEVELS = [2, 3, 4, 5, 6]   # substantive ordinal levels after recode

# Wording-change boundary for USE2_MASK (waves < 22 vs. >= 22)
MASK_ITEM_BREAK_WAVE = 22

# ─ Helper: Shannon entropy (nats) over valid ordinal categories ───────────────
def _shannon_entropy_ordinal(x: pd.Series, valid_levels=None) -> float:
    s = pd.to_numeric(x, errors="coerce").dropna()
    if valid_levels is not None:
        s = s[s.isin(valid_levels)]
    if s.size < 2:
        return 0.0
    p = s.value_counts(normalize=True)
    if p.size < 2:
        return 0.0
    return float(-(p * np.log(p)).sum())

# ─ Helper: max absolute Spearman correlation of b with any column in X ────────
def _max_abs_spearman(b: pd.Series, X: pd.DataFrame, min_n: int = 30) -> float:
    vals = []
    for col in X.columns:
        tmp = pd.concat([b, X[col]], axis=1).dropna()
        if tmp.shape[0] < min_n:
            continue
        r = tmp.iloc[:, 0].corr(tmp.iloc[:, 1], method="spearman")
        if pd.notna(r):
            vals.append(abs(float(r)))
    return float(np.max(vals)) if vals else 0.0

# ─ Compute per-wave raw components ────────────────────────────────────────────
MIN_N_CC = 200   # conservative floor for stable GGM estimation with 8 nodes

rows = []
for w in waves_candidate:
    d    = df.loc[df[WAVE].eq(w), [WAVE] + NODE_VARS + TRUST_VARS].copy()
    d_cc = d.dropna(subset=NODE_VARS)
    n_cc = int(d_cc.shape[0])

    if n_cc < MIN_N_CC:
        rows.append({
            "wave": int(w), "n_cc": n_cc,
            "beh_entropy_mean": np.nan, "beh_coupling_mean": np.nan,
            "score_raw": np.nan, "score_z": np.nan,
            "flag": f"n_cc={n_cc} < {MIN_N_CC} (excluded)",
        })
        continue

    # Shannon entropy per behavior item (valid levels 2–6)
    entropies = [
        _shannon_entropy_ordinal(
            pd.to_numeric(d_cc[b], errors="coerce"),
            valid_levels=BEH_LEVELS,
        )
        for b in BEHAVIOR_VARS
    ]
    beh_entropy_mean = float(np.mean(entropies))

    # Behavior–belief/affect coupling
    X_psych = d_cc[[FOCAL_BELIEF] + AFFECT_VARS].apply(pd.to_numeric, errors="coerce")
    couplings = [
        _max_abs_spearman(pd.to_numeric(d_cc[b], errors="coerce"), X_psych)
        for b in BEHAVIOR_VARS
    ]
    beh_coupling_mean = float(np.mean(couplings))

    rows.append({
        "wave"             : int(w),
        "n_cc"             : n_cc,
        "beh_entropy_mean" : beh_entropy_mean,
        "beh_coupling_mean": beh_coupling_mean,
        "score_raw"        : np.nan,   # filled after z-scoring
        "score_z"          : np.nan,
        "flag"             : "",
    })

wave_rank = pd.DataFrame(rows)

# ─ Z-score each component across eligible waves, then sum ─────────────────────
eligible = wave_rank["flag"].eq("")
for col in ["n_cc", "beh_entropy_mean", "beh_coupling_mean"]:
    vals = wave_rank.loc[eligible, col].values.astype(float)
    if vals.std() > 0:
        wave_rank.loc[eligible, f"_z_{col}"] = zscore(vals)
    else:
        wave_rank.loc[eligible, f"_z_{col}"] = 0.0

wave_rank.loc[eligible, "score_z"] = (
    wave_rank.loc[eligible, "_z_n_cc"] +
    wave_rank.loc[eligible, "_z_beh_entropy_mean"] +
    wave_rank.loc[eligible, "_z_beh_coupling_mean"]
)
# Drop helper columns
wave_rank.drop(columns=[c for c in wave_rank.columns if c.startswith("_z_")],
               inplace=True)

wave_rank = wave_rank.sort_values("score_z", ascending=False, na_position="last")

print("Wave quality ranking (top 20):")
display(wave_rank.head(20))

Wave quality ranking (top 20):


,wave,n_cc,beh_entropy_mean,beh_coupling_mean,score_raw,score_z,flag
46,58,933,1.210077,0.293999,NaN,2.734895,
35,46,945,1.235584,0.272066,NaN,2.606525,
43,55,876,1.293370,0.299130,NaN,2.513983,
45,57,927,1.214008,0.291292,NaN,2.513322,
12,21,942,1.246146,0.257435,NaN,2.105492,
20,30,957,1.109905,0.279900,NaN,1.397169,
34,45,965,1.209776,0.234327,NaN,1.333430,
11,20,925,1.231771,0.255004,NaN,1.303706,
37,48,902,1.266585,0.257326,NaN,1.242234,
38,49,876,1.261713,0.275870,NaN,1.137687,


In [7]:
# ── Select the best consecutive 3-wave block ──────────────────────────────────
# Constraint: the block must NOT straddle the USE2_MASK wording-change boundary
# (wave 22).  All three waves must be < 22 or all three must be >= 22.

rank_lut = (
    wave_rank[wave_rank["flag"].eq("")]
    [["wave", "n_cc", "beh_entropy_mean", "beh_coupling_mean", "score_z"]]
    .set_index("wave")
)

waves_set = set(int(w) for w in waves_candidate if pd.notna(w))

def _straddling_mask_break(w1, w2, w3, break_wave=MASK_ITEM_BREAK_WAVE):
    sides = {w < break_wave for w in (w1, w2, w3)}
    return len(sides) > 1   # True = block crosses the boundary

blocks = []
for w1 in sorted(waves_set):
    w2, w3 = w1 + 1, w1 + 2
    if not {w1, w2, w3}.issubset(waves_set):
        continue
    if not all(w in rank_lut.index for w in (w1, w2, w3)):
        continue   # at least one wave failed the N floor

    if _straddling_mask_break(w1, w2, w3):
        straddle_flag = "straddles USE2_MASK break (wave 22)"
    else:
        straddle_flag = ""

    scores = [float(rank_lut.loc[w, "score_z"]) for w in (w1, w2, w3)]
    ns     = [int(rank_lut.loc[w, "n_cc"])       for w in (w1, w2, w3)]

    if not all(np.isfinite(s) for s in scores):
        continue

    blocks.append({
        "block_start"         : w1,
        "waves"               : [w1, w2, w3],
        "score_z_mean"        : float(np.mean(scores)),
        "score_z_min"         : float(np.min(scores)),
        "n_cc_mean"           : int(np.mean(ns)),
        "n_cc_min"            : int(np.min(ns)),
        "beh_entropy_mean_blk": float(np.mean([
            float(rank_lut.loc[w, "beh_entropy_mean"]) for w in (w1, w2, w3)
        ])),
        "beh_coupling_mean_blk": float(np.mean([
            float(rank_lut.loc[w, "beh_coupling_mean"]) for w in (w1, w2, w3)
        ])),
        "straddle_flag"       : straddle_flag,
    })

if not blocks:
    raise ValueError("No consecutive 3-wave blocks found within waves_candidate. "
                     "Check availability threshold or minimum-N criterion.")

blocks_df = (
    pd.DataFrame(blocks)
    .sort_values(
        by=["straddle_flag", "score_z_mean", "n_cc_min", "score_z_min"],
        ascending=[True, False, False, False],
    )
    .reset_index(drop=True)
)

print("All consecutive 3-wave blocks (ranked):")
display(blocks_df)

# ── Final selection ────────────────────────────────────────────────────────────
# Prefer blocks without the straddle flag; within that, highest score_z_mean.
no_straddle = blocks_df[blocks_df["straddle_flag"].eq("")]
if no_straddle.empty:
    print("\n⚠  All candidate blocks straddle the USE2_MASK wording boundary (wave 22).")
    print("   Proceeding with best-scoring block; interpret USE2_MASK with caution.")
    best_row = blocks_df.iloc[0]
else:
    best_row = no_straddle.iloc[0]

best_block = best_row["waves"]
print(f"\nSelected 3-wave block: {best_block}")
print(f"  score_z_mean  : {best_row['score_z_mean']:.3f}")
print(f"  n_cc_min      : {best_row['n_cc_min']}")
print(f"  straddle_flag : '{best_row['straddle_flag']}'")
print("\nThese wave numbers should be passed to 01_preprocessing as WAVES_SELECTED.")

All consecutive 3-wave blocks (ranked):


,block_start,waves,score_z_mean,score_z_min,n_cc_mean,n_cc_min,beh_entropy_mean_blk,beh_coupling_mean_blk,straddle_flag
0,56,"[56, 57, 58]",1.969140,0.659204,935,927,1.218815,0.269547,
1,55,"[55, 56, 57]",1.895503,0.659204,916,876,1.246580,0.271257,
2,44,"[44, 45, 46]",1.281324,-0.095984,941,914,1.216454,0.248085,
3,54,"[54, 55, 56]",1.002584,-0.165434,899,876,1.270842,0.251796,
4,19,"[19, 20, 21]",0.623060,-1.540018,920,894,1.218855,0.245936,
5,45,"[45, 46, 47]",0.596650,-2.150004,927,872,1.220414,0.239450,
6,46,"[46, 47, 48]",0.566252,-2.150004,906,872,1.239351,0.247117,
7,43,"[43, 44, 45]",0.508199,-0.095984,925,898,1.186022,0.251502,
8,24,"[24, 25, 26]",0.437967,-0.352414,968,962,1.107498,0.247771,
9,16,"[16, 17, 18]",0.359361,0.014274,947,934,1.160864,0.240860,



Selected 3-wave block: [56, 57, 58]
  score_z_mean  : 1.969
  n_cc_min      : 927
  straddle_flag : ''

These wave numbers should be passed to 01_preprocessing as WAVES_SELECTED.
